#### 필요한 라이브러리 불러오기

In [2]:
! pip install selenium

  Using cached selenium-4.27.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached trio_websocket-0.11.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.2.0-py3-none-any.whl.metadata (5.6 kB)
Using cached selenium-4.27.1-py3-none-any.whl (9.7 MB)
Using cached trio_websocket-0.11.1-py3-none-any.whl (17 kB)
Using cached wsproto-1.2.0-py3-none-any.whl (24 kB)
Using cached outcome-1.3.0.post0-py2.py3-none-any.whl (10 kB)
  Attempting uninstall: attrs
    Found existing installation: attrs 23.1.0
    Uninstalling attrs-23.1.0:
      Successfully uninstalled attrs-23.1.0


In [20]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

# 블로그 크롤링에 필요한 라이브러리
from bs4 import BeautifulSoup
import re
import time
import urllib.request
import json
from selenium import webdriver
from selenium.webdriver.common.by import By
import os
#os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-23\bin\server'
#print('JAVA_HOME' in os.environ)

# 형태소 분석에 필요한 라이브러리
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Okt
from collections import Counter

import warnings
warnings.filterwarnings('ignore')

In [13]:
# 웹 드라이버 설정
options = webdriver.ChromeOptions()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# 버전에 상관 없이 os에 설치된 크롬 브라우저 사용
driver = webdriver.Chrome()
driver.implicitly_wait(3)

# Naver API key 입력 : 현재 비식별화 처리 ->  Naver Developers에서 발급 가능
client_id = "8z9rLYIgXT_snHUmoZfA"
client_secret = "dfAmV0fbic"

#### 네이버 API로 원하는 키워드에 해당하는 블로그 url 가져오기|

In [14]:
naver_urls = []
postdate = []
titles = []
data = []
place = []

# 기준 날짜 입력
start_date = input("시작 날짜 (YYYY-MM-DD): ")
end_date = input("종료 날짜 (YYYY-MM-DD): ")

from datetime import datetime
try:
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.strptime(end_date, "%Y-%m-%d")
except ValueError:
    print("날짜 형식이 잘못되었습니다. YYYY-MM-DD 형식으로 입력하세요.")
    exit()

# 검색어 입력
encText = urllib.parse.quote(f"무선스틱청소기")

# 검색을 끝낼 페이지 입력
end = 100
# 한번에 가져올 페이지 입력
display = 100

count = 0
for start in range(end):
    url = "https://openapi.naver.com/v1/search/blog?query=" + encText + "&start=" + str(start * display + 1) + "&display=" + str(display)  # JSON 결과
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()

    if rescode == 200:
        response_body = response.read()
        crawling = json.loads(response_body.decode('utf-8'))['items']

        for row in crawling:
            post_date = row['postdate']  # YYYYMMDD 형식
            post_date_dt = datetime.strptime(post_date, "%Y%m%d")  # 문자열 -> datetime 변환

            # 특정 기간 내 데이터만 추가
            if start_date <= post_date_dt <= end_date and 'blog.naver' in row['link']:
                naver_urls.append(row['link'])
                postdate.append(post_date)

                # html 태그 제거
                title = row['title']
                pattern1 = '<[^>]*>'
                title = re.sub(pattern=pattern1, repl='', string=title)
                titles.append(title)

                data.append(row)
                count += 1  # 조건에 맞는 게시물 카운트 증가

        time.sleep(2)  # API 호출 간 딜레이
    else:
        print("Error Code:" + str(rescode))

print(f"{count}개 크롤링 완료")

HTTPError: HTTP Error 400: Bad Request

In [18]:
len(data)

604

#### 블로그 전문을 추출해 blog.csv파일로 저장

In [16]:
# HTML 태그 제거 함수 정의
def remove_html_tags(text):
    pattern = '<[^>]*>'
    return re.sub(pattern, '', text)

# ConnectionError방지
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/98.0.4758.102"}

contents = []
comments_texts = []
num = 1
try:
    for i in naver_urls:
        print(num, i)
        num += 1
        driver.get(i)
        time.sleep(3)  # 대기시간

        iframe = driver.find_element(By.ID , "mainFrame") 
        driver.switch_to.frame(iframe)

        source = driver.page_source
        html = BeautifulSoup(source, "html.parser")
        
        # 동영상 태그 제거
        for video_tag in html.find_all(["iframe", "video", "div"], class_=re.compile("pzp|video")):
            video_tag.decompose()  # 동영상 관련 태그 제거

        # 기사 텍스트만 가져오기
        content = html.select("div.se-main-container")
        #  list합치기
        content = ''.join(str(content))

        # html태그제거 및 텍스트 다듬기
        content = re.sub(pattern=pattern1, repl='', string=content)
        pattern2 = """[\n\n\n\n\n// flash 오류를 우회하기 위한 함수 추가\nfunction _flash_removeCallback() {}"""
        content = content.replace(pattern2, '')
        content = content.replace('\n', '')
        content = content.replace('\u200b', '')
        contents.append(content)

    # 결과 저장
    news_df = pd.DataFrame({'title': titles, 'content': contents, 'date': postdate, 'link': naver_urls})
    news_df['content'] = news_df['content'].apply(remove_html_tags)
    news_df.to_csv('무선스틱청소기.csv', index=False, encoding='utf-8-sig')
except:
    contents.append('error')
    news_df = pd.DataFrame({'title': titles, 'content': contents, 'date': postdate, 'link': naver_urls})
    news_df['content'] = news_df['content'].apply(remove_html_tags)
    news_df.to_csv('무선스틱청소기.csv', index=False, encoding='utf-8-sig')

1 https://blog.naver.com/joy0090/223691040550
2 https://blog.naver.com/dean4001/223383394064
3 https://blog.naver.com/jo6699/223644094789
4 https://blog.naver.com/ardeur80/223566115696
5 https://blog.naver.com/hongmine2/223501465668
6 https://blog.naver.com/temis930/223703867810
7 https://blog.naver.com/vvqbgteds/223698498919
8 https://blog.naver.com/sm_511/223692571242
9 https://blog.naver.com/orwrbb2546/223641863048
10 https://blog.naver.com/alwaysbetrue_/223665385707
11 https://blog.naver.com/5s8s59kul/223678369423
12 https://blog.naver.com/shl2n2o4/223678457028
13 https://blog.naver.com/ttvwnr2455/223582256387
14 https://blog.naver.com/spgk2/223658575645
15 https://blog.naver.com/xlqocw4808/223599026859
16 https://blog.naver.com/ccgu4745/223705876999
17 https://blog.naver.com/sayye_s/223685256548
18 https://blog.naver.com/temis930/223703871948
19 https://blog.naver.com/pvhfzi3834/223599250586
20 https://blog.naver.com/vvqbgteds/223698609832
21 https://blog.naver.com/gposzp6765/2235